<a href="https://colab.research.google.com/github/rhigazi/pmTool/blob/main/PMTool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!npm install -g @opencode/cli

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 5 packages in 24s
⠴npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.2
npm notice To update run: npm install -g npm@12.2.2
npm notice
⠴

In [2]:
# @title Zelle 6: Telegram-Benachrichtigung (Abschluss)
import datetime
import os
import sys

import requests

# --- Laufzeit-Check & Secrets (Dual-Environment wie in Zelle 1/2) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = userdata.get("TELEGRAM_CHAT_ID")
else:
    TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    print("Telegram-Secrets fehlen (TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID) – Benachrichtigung übersprungen.")
else:
    # --- Kurze Zusammenfassung für die Nachricht ---
    WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())
    _okf_dir = os.path.join(WORKDIR, "okf-regeln")
    _okf_count = len([f for f in os.listdir(_okf_dir) if f.endswith(".md")]) if os.path.isdir(_okf_dir) else 0
    _erwartung = "erstellt/aktualisiert" if os.path.exists(os.path.join(WORKDIR, "erwartung.md")) else "fehlt"
    _rows = len(df) if "df" in globals() else "?"
    _stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    text = f"""✅ Newscheck-Pipeline abgeschlossen ({_stamp})

• Marktdaten (DIA): {_rows} Zeilen
• OKF-Regeln: {_okf_count} Dateien in okf-regeln/
• erwartung.md: {_erwartung}
• Push nach origin/main: siehe vorherige Zelle"""

    # --- Telegram senden (best effort; Versandfehler sind nicht fatal) ---
    _url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        _resp = requests.post(_url, data={"chat_id": TELEGRAM_CHAT_ID, "text": text}, timeout=30)
        if _resp.status_code == 200 and _resp.json().get("ok"):
            print("Telegram-Nachricht gesendet ✓")
        else:
            print("Telegram-Versand fehlgeschlagen:", _resp.status_code, _resp.text)
    except requests.RequestException as e:
        print("Telegram-Versand fehlgeschlagen (Netzwerkfehler):", e)


Telegram-Nachricht gesendet ✓


In [3]:
# @title Telegram-Secrets → /content/config.json (für telegram_send.py / Agent)
import json
import sys
from pathlib import Path

# --- Nur in Colab sinnvoll ---
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Diese Zelle ist für Google Colab gedacht (google.colab.userdata).")

from google.colab import userdata

# --- Secrets lesen ---
bot_token = userdata.get("TELEGRAM_BOT_TOKEN")
chat_id = userdata.get("TELEGRAM_CHAT_ID")

# --- Validierung ---
missing = [k for k, v in {
    "TELEGRAM_BOT_TOKEN": bot_token,
    "TELEGRAM_CHAT_ID": chat_id,
}.items() if not v]
if missing:
    raise ValueError(f"Fehlende Colab-Secrets: {', '.join(missing)}")

# chat_id als String speichern (telegram_send.py akzeptiert String/Zahl)
config = {
    "bot_token": str(bot_token).strip(),
    "chat_id": str(chat_id).strip(),
}

# --- Nach /content schreiben (Projektroot, von telegram_send.py erwartet) ---
target = Path("/content/config.json")
target.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# --- Kurze Bestätigung ohne Secrets preiszugeben ---
def _mask(s: str, keep: int = 4) -> str:
    s = str(s)
    return s[:keep] + "…" if len(s) > keep else "…"

print(f"✓ Konfiguration geschrieben: {target}")
print(f"  bot_token: {_mask(config['bot_token'])} (gesetzt)")
print(f"  chat_id:   {_mask(config['chat_id'])} (gesetzt)")
print("Hinweis: /content/config.json steht in .gitignore und wird nicht committet.")

✓ Konfiguration geschrieben: /content/config.json
  bot_token: 8686… (gesetzt)
  chat_id:   7482… (gesetzt)
Hinweis: /content/config.json steht in .gitignore und wird nicht committet.


In [4]:
from google.colab import userdata
import os

# 1. Token aus den Colab Secrets auslesen
github_token = userdata.get('GITHUB_TOKEN')

# 2. Git-Identität konfigurieren (bitte anpassen)
!git config --global user.name "rhigazi"
!git config --global user.email "rhigazi@gmail.com"

# 3. Repository klonen (Authentifizierung erfolgt über den Token in der URL)
repo_url = f"https://{github_token}@github.com/rhigazi/pmTool.git"

if not os.path.exists("pmTool"):
    !git clone {repo_url}
    print("Erfolgreich authentifiziert und Repository geklont!")
else:
    print("Repository 'pmTool' existiert bereits lokal.")

Cloning into 'pmTool'...
Erfolgreich authentifiziert und Repository geklont!
